In [4]:
# =============================================================================
# Cell 0
# Validation Metrics Notebook
#
# Purpose
# --------
# Load all datasets required to validate the packaged RozviDrought SPI3 model
# against the historical Excel drought catalogue.
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau,
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import matplotlib.pyplot as plt

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd()

DATA_DIR = (
    NOTEBOOK_DIR.parents[1]
    / "data"
    / "validation"
)

OUTPUT_DIR = DATA_DIR / "outputs"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# -----------------------------------------------------------------------------
# Input files
# -----------------------------------------------------------------------------

EXCEL_LEDGER = DATA_DIR / "excel_event_ledger.csv"

MODEL_EVENT_SUMMARY = DATA_DIR / "model_event_summary.csv"

MONTHLY_EVENT_SUMMARY = DATA_DIR / "monthly_event_summary.csv"

ORDINAL_RANKING = DATA_DIR / "ordinal_ranking.csv"

ROZVIDROUGHT_EVENT_SUMMARY = (
    DATA_DIR
    / "rozvidrought_spi3_event_summary.csv"
)

ROZVIDROUGHT_MONTHLY_SUMMARY = (
    DATA_DIR
    / "rozvidrought_spi3_monthly_summary.csv"
)

ROZVIDROUGHT_PIXEL_PREDICTIONS = (
    DATA_DIR
    / "rozvidrought_spi3_pixel_predictions.parquet"
)

ROZVIDROUGHT_MANIFEST = (
    DATA_DIR
    / "rozvidrought_spi3_manifest.json"
)

# -----------------------------------------------------------------------------
# Verify files
# -----------------------------------------------------------------------------

required_files = {
    "excel_event_ledger": EXCEL_LEDGER,
    "model_event_summary": MODEL_EVENT_SUMMARY,
    "monthly_event_summary": MONTHLY_EVENT_SUMMARY,
    "ordinal_ranking": ORDINAL_RANKING,
    "rozvidrought_event_summary": ROZVIDROUGHT_EVENT_SUMMARY,
    "rozvidrought_monthly_summary": ROZVIDROUGHT_MONTHLY_SUMMARY,
    "rozvidrought_pixel_predictions": ROZVIDROUGHT_PIXEL_PREDICTIONS,
    "rozvidrought_manifest": ROZVIDROUGHT_MANIFEST,
}

missing = [
    name
    for name, path in required_files.items()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing files:\n"
        + "\n".join(missing)
    )

# -----------------------------------------------------------------------------
# Load
# -----------------------------------------------------------------------------

excel_df = pd.read_csv(EXCEL_LEDGER)

model_event_df = pd.read_csv(MODEL_EVENT_SUMMARY)

monthly_event_df = pd.read_csv(MONTHLY_EVENT_SUMMARY)

ordinal_df = pd.read_csv(ORDINAL_RANKING)

rozvidrought_event_df = pd.read_csv(
    ROZVIDROUGHT_EVENT_SUMMARY
)

rozvidrought_monthly_df = pd.read_csv(
    ROZVIDROUGHT_MONTHLY_SUMMARY
)

rozvidrought_pixel_df = pd.read_parquet(
    ROZVIDROUGHT_PIXEL_PREDICTIONS
)

# -----------------------------------------------------------------------------
# Summary
# -----------------------------------------------------------------------------

print("=" * 80)
print("VALIDATION DATA LOADED")
print("=" * 80)

print()

for name, df in [
    ("Excel event ledger", excel_df),
    ("Current model event summary", model_event_df),
    ("Current model monthly summary", monthly_event_df),
    ("Current model ordinal ranking", ordinal_df),
    ("RozviDrought event summary", rozvidrought_event_df),
    ("RozviDrought monthly summary", rozvidrought_monthly_df),
    ("RozviDrought pixel predictions", rozvidrought_pixel_df),
]:
    print(
        f"{name:<35}"
        f"Rows={len(df):>10,}"
        f"   Columns={len(df.columns):>3}"
    )

print()

print("Validation directory:")
print(DATA_DIR)

print()

print("Output directory:")
print(OUTPUT_DIR)

VALIDATION DATA LOADED

Excel event ledger                 Rows=        17   Columns= 18
Current model event summary        Rows=         3   Columns= 26
Current model monthly summary      Rows=        19   Columns= 34
Current model ordinal ranking      Rows=         1   Columns=  4
RozviDrought event summary         Rows=         3   Columns=  4
RozviDrought monthly summary       Rows=        19   Columns=  5
RozviDrought pixel predictions     Rows=   508,725   Columns= 12

Validation directory:
c:\Projects\Infer RozviDrought\data\validation

Output directory:
c:\Projects\Infer RozviDrought\data\validation\outputs


In [5]:
# =============================================================================
# Cell 1
# Build event comparison table
#
# Purpose
# --------
# Align the historical Excel catalogue with the packaged RozviDrought
# (SPI3 subsystem-fusion) event summaries.
# =============================================================================

# -----------------------------------------------------------------------------
# Standardise event names
# -----------------------------------------------------------------------------

excel = excel_df.copy()

excel["event_key"] = (
    "zimbabwe_"
    + excel["event_period_start_year"].astype(str)
    + "_"
    + excel["event_period_end_year"].astype(str)
)

# -----------------------------------------------------------------------------
# Rename RozviDrought summary columns
# -----------------------------------------------------------------------------

rv = rozvidrought_event_df.rename(
    columns={
        "event_id": "event_key",
        "mean_class": "rozvidrought_mean_class",
        "mean_confidence": "rozvidrought_mean_confidence",
        "mean_p3": "rozvidrought_mean_p3",
    }
)

# -----------------------------------------------------------------------------
# Merge
# -----------------------------------------------------------------------------

comparison_df = (
    excel.merge(
        rv,
        on="event_key",
        how="left",
    )
    .sort_values(
        "event_period_start_year"
    )
    .reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# Keep useful columns
# -----------------------------------------------------------------------------

comparison_df = comparison_df[
    [
        "event_key",
        "excel_event_id",
        "event_period_raw",
        "duration_months",
        "excel_severity_score",
        "excel_severity_label",
        "event_start_yyyymm",
        "event_end_yyyymm",
        "event_month_count",
        "rozvidrought_mean_class",
        "rozvidrought_mean_confidence",
        "rozvidrought_mean_p3",
    ]
]

comparison_df["validated"] = (
    comparison_df["rozvidrought_mean_class"]
    .notna()
)

# -----------------------------------------------------------------------------
# Save
# -----------------------------------------------------------------------------

comparison_path = (
    OUTPUT_DIR
    / "event_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False,
)

# -----------------------------------------------------------------------------
# Summary
# -----------------------------------------------------------------------------

print("=" * 80)
print("EVENT COMPARISON")
print("=" * 80)

print()

print(
    "Historical events :",
    len(comparison_df),
)

print(
    "Validated events  :",
    comparison_df["validated"].sum(),
)

print(
    "Pending events    :",
    (~comparison_df["validated"]).sum(),
)

print()

print(
    comparison_df[
        comparison_df["validated"]
    ].to_string(index=False)
)

print()

print("Saved:")
print(comparison_path)

EVENT COMPARISON

Historical events : 17
Validated events  : 3
Pending events    : 14

         event_key  excel_event_id event_period_raw  duration_months  excel_severity_score excel_severity_label  event_start_yyyymm  event_end_yyyymm  event_month_count  rozvidrought_mean_class  rozvidrought_mean_confidence  rozvidrought_mean_p3  validated
zimbabwe_2015_2016              15        2015–2016               12                     4          4 – Extreme              201510            201609                 12                 0.155757                      0.944111              0.003147       True
zimbabwe_2018_2019              16        2018–2019                8                     2         2 – Moderate              201810            201905                  8                 0.118058                      0.954125              0.000937       True
zimbabwe_2023_2024              17        2023–2024               12                     5     5 – Catastrophic              202310           

In [6]:
# =============================================================================
# Cell 2
# Quantitative validation metrics
#
# Purpose
# --------
# Compare the packaged RozviDrought SPI3 model against the Excel drought
# catalogue using the validated historical events.
# =============================================================================

metrics_df = (
    comparison_df.loc[
        comparison_df["validated"]
    ]
    .copy()
)

print("=" * 80)
print("VALIDATION METRICS")
print("=" * 80)

# -----------------------------------------------------------------------------
# Variables
# -----------------------------------------------------------------------------

excel = (
    metrics_df["excel_severity_score"]
    .astype(float)
    .to_numpy()
)

model = (
    metrics_df["rozvidrought_mean_class"]
    .astype(float)
    .to_numpy()
)

# -----------------------------------------------------------------------------
# Correlations
# -----------------------------------------------------------------------------

pearson_r, pearson_p = pearsonr(
    excel,
    model,
)

spearman_rho, spearman_p = spearmanr(
    excel,
    model,
)

kendall_tau, kendall_p = kendalltau(
    excel,
    model,
)

# -----------------------------------------------------------------------------
# Scale model to Excel range (1-5)
# -----------------------------------------------------------------------------

model_scaled = (
    1
    + (model / 4.0) * 4.0
)

# -----------------------------------------------------------------------------
# Error metrics
# -----------------------------------------------------------------------------

mae = mean_absolute_error(
    excel,
    model_scaled,
)

rmse = np.sqrt(
    mean_squared_error(
        excel,
        model_scaled,
    )
)

r2 = r2_score(
    excel,
    model_scaled,
)

# -----------------------------------------------------------------------------
# Rank accuracy
# -----------------------------------------------------------------------------

excel_rank = (
    metrics_df["excel_severity_score"]
    .rank(
        ascending=False,
        method="dense",
    )
    .astype(int)
)

model_rank = (
    metrics_df["rozvidrought_mean_class"]
    .rank(
        ascending=False,
        method="dense",
    )
    .astype(int)
)

rank_accuracy = (
    (excel_rank == model_rank)
    .mean()
    * 100
)

top_accuracy = int(
    excel_rank.idxmin()
    ==
    model_rank.idxmin()
)

bottom_accuracy = int(
    excel_rank.idxmax()
    ==
    model_rank.idxmax()
)

metrics_df["excel_rank"] = excel_rank
metrics_df["model_rank"] = model_rank
metrics_df["rank_difference"] = (
    model_rank
    - excel_rank
)

# -----------------------------------------------------------------------------
# Metrics table
# -----------------------------------------------------------------------------

metrics_table = pd.DataFrame(
    {
        "metric": [
            "Pearson r",
            "Spearman rho",
            "Kendall tau",
            "MAE",
            "RMSE",
            "R2",
            "Rank accuracy (%)",
            "Top-event accuracy",
            "Weakest-event accuracy",
        ],
        "value": [
            pearson_r,
            spearman_rho,
            kendall_tau,
            mae,
            rmse,
            r2,
            rank_accuracy,
            top_accuracy,
            bottom_accuracy,
        ],
    }
)

# -----------------------------------------------------------------------------
# Save
# -----------------------------------------------------------------------------

metrics_path = (
    OUTPUT_DIR
    / "validation_metrics.csv"
)

ranking_path = (
    OUTPUT_DIR
    / "event_rankings.csv"
)

metrics_table.to_csv(
    metrics_path,
    index=False,
)

metrics_df.to_csv(
    ranking_path,
    index=False,
)

# -----------------------------------------------------------------------------
# Print
# -----------------------------------------------------------------------------

print()
print(metrics_table.to_string(index=False))

print()
print("=" * 80)
print("EVENT RANKING")
print("=" * 80)

print()

print(
    metrics_df[
        [
            "event_period_raw",
            "excel_severity_score",
            "rozvidrought_mean_class",
            "excel_rank",
            "model_rank",
            "rank_difference",
        ]
    ].to_string(index=False)
)

print()

print("Saved:")
print(metrics_path)
print(ranking_path)

VALIDATION METRICS

                metric     value
             Pearson r -0.303378
          Spearman rho -0.500000
           Kendall tau -0.333333
                   MAE  2.547556
                  RMSE  2.840562
                    R2 -4.187081
     Rank accuracy (%)  0.000000
    Top-event accuracy  0.000000
Weakest-event accuracy  0.000000

EVENT RANKING

event_period_raw  excel_severity_score  rozvidrought_mean_class  excel_rank  model_rank  rank_difference
       2015–2016                     4                 0.155757           2           1               -1
       2018–2019                     2                 0.118058           3           2               -1
       2023–2024                     5                 0.083517           1           3                2

Saved:
c:\Projects\Infer RozviDrought\data\validation\outputs\validation_metrics.csv
c:\Projects\Infer RozviDrought\data\validation\outputs\event_rankings.csv


In [7]:
# =============================================================================
# Cell 3
# Reconstruct event-level severity from RozviDrought pixel predictions
#
# Purpose
# --------
# Produce event metrics comparable to the newer fusion evaluation using the
# original packaged RozviDrought outputs.
# =============================================================================

print("=" * 80)
print("RECONSTRUCTING ROZVIDROUGHT EVENT SEVERITY")
print("=" * 80)

pixel_df = rozvidrought_pixel_df.copy()

# -----------------------------------------------------------------------------
# Standardise month column
# -----------------------------------------------------------------------------

if "yyyymm" not in pixel_df.columns:
    raise KeyError(
        f"Could not find yyyymm column.\nColumns:\n{pixel_df.columns.tolist()}"
    )

pixel_df["yyyymm"] = (
    pd.to_numeric(pixel_df["yyyymm"])
)

# -----------------------------------------------------------------------------
# Locate prediction columns
# -----------------------------------------------------------------------------

prob_cols = [
    c
    for c in pixel_df.columns
    if c.lower() in {"p0", "p1", "p2", "p3"}
]

if len(prob_cols) != 4:
    raise RuntimeError(
        f"Expected probability columns p0-p3.\nFound: {prob_cols}"
    )

print()
print("Probability columns:", prob_cols)

# -----------------------------------------------------------------------------
# Expected severity
# -----------------------------------------------------------------------------

weights = np.arange(4, dtype=float)

pixel_df["expected_class"] = (
    pixel_df[prob_cols].to_numpy()
    @ weights
)

pixel_df["drought_probability"] = (
    pixel_df["p2"]
    + pixel_df["p3"]
)

pixel_df["severe_probability"] = (
    pixel_df["p3"]
)

pixel_df["predicted_class"] = (
    pixel_df[prob_cols]
    .to_numpy()
    .argmax(axis=1)
)

# -----------------------------------------------------------------------------
# Event summaries
# -----------------------------------------------------------------------------

event_rows = []

for _, event in excel_df.iterrows():

    subset = pixel_df.loc[
        (
            pixel_df["yyyymm"]
            >= event["event_start_yyyymm"]
        )
        &
        (
            pixel_df["yyyymm"]
            <= event["event_end_yyyymm"]
        )
    ].copy()

    if subset.empty:
        continue

    monthly = (
        subset
        .groupby("yyyymm")
        .agg(
            mean_expected_class=(
                "expected_class",
                "mean",
            ),
            mean_drought_probability=(
                "drought_probability",
                "mean",
            ),
            mean_p3=(
                "p3",
                "mean",
            ),
            severe_share=(
                "predicted_class",
                lambda x: (x >= 3).mean(),
            ),
        )
        .reset_index()
    )

    event_rows.append(
        {
            "event_key":
                event["event_key"],

            "excel_event_id":
                event["excel_event_id"],

            "event_period":
                event["event_period_raw"],

            "excel_severity":
                event["excel_severity_score"],

            "window_start":
                event["event_start_yyyymm"],

            "window_end":
                event["event_end_yyyymm"],

            "months":
                len(monthly),

            "mean_expected_class":
                monthly["mean_expected_class"].mean(),

            "peak_expected_class":
                monthly["mean_expected_class"].max(),

            "peak_expected_month":
                monthly.loc[
                    monthly["mean_expected_class"].idxmax(),
                    "yyyymm",
                ],

            "mean_drought_probability":
                monthly["mean_drought_probability"].mean(),

            "peak_drought_probability":
                monthly["mean_drought_probability"].max(),

            "peak_probability_month":
                monthly.loc[
                    monthly["mean_drought_probability"].idxmax(),
                    "yyyymm",
                ],

            "mean_p3":
                monthly["mean_p3"].mean(),

            "peak_p3":
                monthly["mean_p3"].max(),

            "mean_severe_share":
                monthly["severe_share"].mean(),
        }
    )

rozvi_event_metrics = pd.DataFrame(event_rows)

# -----------------------------------------------------------------------------
# Restrict to events with predictions
# -----------------------------------------------------------------------------

rozvi_event_metrics = (
    rozvi_event_metrics
    .loc[
        rozvi_event_metrics["mean_expected_class"].notna()
    ]
    .reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# Save
# -----------------------------------------------------------------------------

output_path = (
    OUTPUT_DIR
    / "rozvidrought_reconstructed_event_metrics.csv"
)

rozvi_event_metrics.to_csv(
    output_path,
    index=False,
)

# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

print()

print(
    rozvi_event_metrics[
        [
            "event_period",
            "excel_severity",
            "mean_expected_class",
            "peak_expected_class",
            "mean_drought_probability",
            "peak_drought_probability",
            "mean_severe_share",
        ]
    ].to_string(index=False)
)

print()

print("Saved:")
print(output_path)

RECONSTRUCTING ROZVIDROUGHT EVENT SEVERITY

Probability columns: ['p0', 'p1', 'p2', 'p3']

event_period  excel_severity  mean_expected_class  peak_expected_class  mean_drought_probability  peak_drought_probability  mean_severe_share
   2015–2016               4             0.186782             0.796420                  0.034361                  0.160105           0.001008
   2018–2019               2             0.140080             0.914113                  0.037035                  0.282133           0.000103
   2023–2024               5             0.101787             0.489493                  0.004549                  0.019875           0.000006

Saved:
c:\Projects\Infer RozviDrought\data\validation\outputs\rozvidrought_reconstructed_event_metrics.csv


In [8]:
# =============================================================================
# Cell 4
# Quantitative validation using reconstructed event severity
# =============================================================================

print("=" * 80)
print("ROZVIDROUGHT VALIDATION METRICS")
print("=" * 80)

metrics_df = rozvi_event_metrics.copy()

# -----------------------------------------------------------------------------
# Variables
# -----------------------------------------------------------------------------

excel = metrics_df["excel_severity"].to_numpy(dtype=float)

model = metrics_df["mean_expected_class"].to_numpy(dtype=float)

# Scale expected class (0-4) -> Excel scale (1-5)
model_scaled = model + 1.0

# -----------------------------------------------------------------------------
# Correlations
# -----------------------------------------------------------------------------

pearson_r, pearson_p = pearsonr(
    excel,
    model,
)

spearman_rho, spearman_p = spearmanr(
    excel,
    model,
)

kendall_tau, kendall_p = kendalltau(
    excel,
    model,
)

# -----------------------------------------------------------------------------
# Error metrics
# -----------------------------------------------------------------------------

mae = mean_absolute_error(
    excel,
    model_scaled,
)

rmse = np.sqrt(
    mean_squared_error(
        excel,
        model_scaled,
    )
)

r2 = r2_score(
    excel,
    model_scaled,
)

# -----------------------------------------------------------------------------
# Ranking
# -----------------------------------------------------------------------------

metrics_df["excel_rank"] = (
    metrics_df["excel_severity"]
    .rank(
        ascending=False,
        method="dense",
    )
    .astype(int)
)

metrics_df["model_rank"] = (
    metrics_df["mean_expected_class"]
    .rank(
        ascending=False,
        method="dense",
    )
    .astype(int)
)

metrics_df["rank_difference"] = (
    metrics_df["model_rank"]
    - metrics_df["excel_rank"]
)

rank_accuracy = (
    (
        metrics_df["excel_rank"]
        ==
        metrics_df["model_rank"]
    )
    .mean()
    * 100
)

top_accuracy = float(
    metrics_df.loc[
        metrics_df["excel_rank"].idxmin(),
        "model_rank",
    ]
    == 1
)

bottom_accuracy = float(
    metrics_df.loc[
        metrics_df["excel_rank"].idxmax(),
        "model_rank",
    ]
    == metrics_df["model_rank"].max()
)

# -----------------------------------------------------------------------------
# Metrics table
# -----------------------------------------------------------------------------

metrics_table = pd.DataFrame(
    {
        "Metric": [
            "Pearson correlation",
            "Spearman correlation",
            "Kendall Tau",
            "MAE",
            "RMSE",
            "R²",
            "Rank accuracy (%)",
            "Top event accuracy",
            "Weakest event accuracy",
        ],
        "Value": [
            pearson_r,
            spearman_rho,
            kendall_tau,
            mae,
            rmse,
            r2,
            rank_accuracy,
            top_accuracy,
            bottom_accuracy,
        ],
    }
)

# -----------------------------------------------------------------------------
# Save
# -----------------------------------------------------------------------------

metrics_table.to_csv(
    OUTPUT_DIR / "rozvidrought_validation_metrics.csv",
    index=False,
)

metrics_df.to_csv(
    OUTPUT_DIR / "rozvidrought_event_rankings.csv",
    index=False,
)

# -----------------------------------------------------------------------------
# Print
# -----------------------------------------------------------------------------

print()
print(metrics_table.to_string(index=False))

print()
print("=" * 80)
print("EVENT RANKING")
print("=" * 80)

print()

print(
    metrics_df[
        [
            "event_period",
            "excel_severity",
            "mean_expected_class",
            "excel_rank",
            "model_rank",
            "rank_difference",
        ]
    ].to_string(index=False)
)

print()

print("Saved:")
print(OUTPUT_DIR / "rozvidrought_validation_metrics.csv")
print(OUTPUT_DIR / "rozvidrought_event_rankings.csv")

ROZVIDROUGHT VALIDATION METRICS

                Metric     Value
   Pearson correlation -0.272909
  Spearman correlation -0.500000
           Kendall Tau -0.333333
                   MAE  2.523783
                  RMSE  2.819558
                    R² -4.110654
     Rank accuracy (%)  0.000000
    Top event accuracy  0.000000
Weakest event accuracy  0.000000

EVENT RANKING

event_period  excel_severity  mean_expected_class  excel_rank  model_rank  rank_difference
   2015–2016               4             0.186782           2           1               -1
   2018–2019               2             0.140080           3           2               -1
   2023–2024               5             0.101787           1           3                2

Saved:
c:\Projects\Infer RozviDrought\data\validation\outputs\rozvidrought_validation_metrics.csv
c:\Projects\Infer RozviDrought\data\validation\outputs\rozvidrought_event_rankings.csv


In [9]:
# =============================================================================
# Cell 5
# Reconstruct event severity using percentile-based aggregation
#
# Motivation
# ----------
# The Excel catalogue reflects impactful droughts rather than the national
# average state. Therefore compute severity from the most drought-affected
# pixels rather than averaging every grid cell.
# =============================================================================

print("=" * 80)
print("PERCENTILE-BASED EVENT SEVERITY")
print("=" * 80)

pixel_df = rozvidrought_pixel_df.copy()

pixel_df["expected_class"] = (
    pixel_df[["p0","p1","p2","p3"]].to_numpy()
    @ np.arange(4)
)

pixel_df["drought_probability"] = (
    pixel_df["p2"]
    + pixel_df["p3"]
)

pixel_df["yyyymm"] = pd.to_numeric(pixel_df["yyyymm"])

TOP_PERCENTILES = [
    0.01,
    0.05,
    0.10,
]

event_results = []

for _, event in excel_df.iterrows():

    event_pixels = pixel_df.loc[
        (pixel_df["yyyymm"] >= event["event_start_yyyymm"])
        &
        (pixel_df["yyyymm"] <= event["event_end_yyyymm"])
    ].copy()

    if event_pixels.empty:
        continue

    monthly_results = []

    for month, month_df in event_pixels.groupby("yyyymm"):

        row = {
            "yyyymm": month,
        }

        row["national_mean"] = (
            month_df["expected_class"].mean()
        )

        row["national_peak"] = (
            month_df["expected_class"].max()
        )

        for pct in TOP_PERCENTILES:

            n = max(
                1,
                int(len(month_df) * pct),
            )

            top = (
                month_df
                .nlargest(
                    n,
                    "expected_class",
                )
            )

            row[f"top_{int(pct*100)}_mean"] = (
                top["expected_class"].mean()
            )

            row[f"top_{int(pct*100)}_peak"] = (
                top["expected_class"].max()
            )

            row[f"top_{int(pct*100)}_drought_probability"] = (
                top["drought_probability"].mean()
            )

        monthly_results.append(row)

    monthly_results = pd.DataFrame(monthly_results)

    summary = {
        "event_key":
            event["event_key"],

        "event_period":
            event["event_period_raw"],

        "excel_severity":
            event["excel_severity_score"],
    }

    for col in monthly_results.columns:

        if col == "yyyymm":
            continue

        summary[col] = (
            monthly_results[col].mean()
        )

    event_results.append(summary)

event_percentiles = pd.DataFrame(event_results)

# -----------------------------------------------------------------------------
# Ranking diagnostics
# -----------------------------------------------------------------------------

ranking_cols = [
    "national_mean",
    "top_1_mean",
    "top_5_mean",
    "top_10_mean",
]

print()

print("=" * 80)
print("RANKING COMPARISON")
print("=" * 80)

for col in ranking_cols:

    if col not in event_percentiles.columns:
        continue

    ranked = (
        event_percentiles[
            [
                "event_period",
                "excel_severity",
                col,
            ]
        ]
        .sort_values(
            col,
            ascending=False,
        )
    )

    print()
    print(col)
    print(ranked.to_string(index=False))

# -----------------------------------------------------------------------------
# Save
# -----------------------------------------------------------------------------

output = (
    OUTPUT_DIR
    / "rozvidrought_percentile_event_metrics.csv"
)

event_percentiles.to_csv(
    output,
    index=False,
)

print()
print("Saved:")
print(output)

PERCENTILE-BASED EVENT SEVERITY

RANKING COMPARISON

national_mean
event_period  excel_severity  national_mean
   2015–2016               4       0.186782
   2018–2019               2       0.140080
   2023–2024               5       0.101787

top_1_mean
event_period  excel_severity  top_1_mean
   2015–2016               4    0.706684
   2018–2019               2    0.535546
   2023–2024               5    0.453558

top_5_mean
event_period  excel_severity  top_5_mean
   2015–2016               4    0.557929
   2018–2019               2    0.401741
   2023–2024               5    0.310129

top_10_mean
event_period  excel_severity  top_10_mean
   2015–2016               4     0.491604
   2018–2019               2     0.355859
   2023–2024               5     0.267755

Saved:
c:\Projects\Infer RozviDrought\data\validation\outputs\rozvidrought_percentile_event_metrics.csv


In [10]:
# =============================================================================
# Cell 6
# Monthly chronology validation
#
# Purpose
# --------
# Evaluate whether RozviDrought captures the temporal evolution of each event,
# rather than only comparing overall event severity.
# =============================================================================

print("=" * 80)
print("MONTHLY CHRONOLOGY VALIDATION")
print("=" * 80)

monthly = rozvidrought_monthly_df.copy()
monthly["yyyymm"] = pd.to_numeric(monthly["yyyymm"])

results = []

for _, event in excel_df.iterrows():

    event_months = monthly.loc[
        (monthly["yyyymm"] >= event["event_start_yyyymm"])
        &
        (monthly["yyyymm"] <= event["event_end_yyyymm"])
    ].copy()

    if event_months.empty:
        continue

    peak_idx = event_months["mean_class"].idxmax()

    peak_month = int(
        event_months.loc[
            peak_idx,
            "yyyymm",
        ]
    )

    peak_class = float(
        event_months.loc[
            peak_idx,
            "mean_class",
        ]
    )

    mean_class = float(
        event_months["mean_class"].mean()
    )

    persistence = float(
        (
            event_months["mean_class"]
            >= mean_class
        ).mean()
    )

    first_month = int(
        event_months["yyyymm"].min()
    )

    last_month = int(
        event_months["yyyymm"].max()
    )

    peak_offset = len(
        event_months.loc[
            event_months["yyyymm"] < peak_month
        ]
    )

    results.append(
        {
            "event_key": event["event_key"],
            "event_period": event["event_period_raw"],
            "excel_severity": event["excel_severity_score"],
            "months_available": len(event_months),
            "first_month": first_month,
            "last_month": last_month,
            "peak_month": peak_month,
            "peak_offset_months": peak_offset,
            "mean_class": mean_class,
            "peak_class": peak_class,
            "persistence_score": persistence,
        }
    )

chronology_df = pd.DataFrame(results)

chronology_df.to_csv(
    OUTPUT_DIR / "rozvidrought_monthly_chronology.csv",
    index=False,
)

print()
print(
    chronology_df.to_string(index=False)
)

print()

print("Saved:")
print(
    OUTPUT_DIR / "rozvidrought_monthly_chronology.csv"
)

MONTHLY CHRONOLOGY VALIDATION

               event_key event_period  excel_severity  months_available  first_month  last_month  peak_month  peak_offset_months  mean_class  peak_class  persistence_score
excel_event_15_2015–2016    2015–2016               4                 5       201510      201602      201510                   0    0.155757    0.757199           0.200000
excel_event_16_2018–2019    2018–2019               2                 8       201810      201905      201810                   0    0.118058    0.923324           0.125000
excel_event_17_2023–2024    2023–2024               5                 6       202310      202403      202310                   0    0.083517    0.486312           0.166667

Saved:
c:\Projects\Infer RozviDrought\data\validation\outputs\rozvidrought_monthly_chronology.csv


In [11]:
# =============================================================================
# Cell 7
# New Multi-Horizon Model Validation Metrics
#
# Purpose
# --------
# Evaluate the new SPI3/SPI6/SPI12 fusion model using the same methodology
# applied to RozviDrought so that both models can be compared fairly.
# =============================================================================

print("=" * 80)
print("NEW MULTI-HORIZON MODEL VALIDATION")
print("=" * 80)

fusion = model_event_df.copy()

# -----------------------------------------------------------------------------
# Variables
# -----------------------------------------------------------------------------

excel = fusion["excel_severity_score"].astype(float).to_numpy()

model = (
    fusion["fusion_event_mean_expected_class_0_to_4"]
    .astype(float)
    .to_numpy()
)

model_scaled = model + 1.0

# -----------------------------------------------------------------------------
# Correlations
# -----------------------------------------------------------------------------

pearson_r, pearson_p = pearsonr(
    excel,
    model,
)

spearman_rho, spearman_p = spearmanr(
    excel,
    model,
)

kendall_tau, kendall_p = kendalltau(
    excel,
    model,
)

# -----------------------------------------------------------------------------
# Error metrics
# -----------------------------------------------------------------------------

mae = mean_absolute_error(
    excel,
    model_scaled,
)

rmse = np.sqrt(
    mean_squared_error(
        excel,
        model_scaled,
    )
)

r2 = r2_score(
    excel,
    model_scaled,
)

# -----------------------------------------------------------------------------
# Ranking
# -----------------------------------------------------------------------------

fusion["excel_rank"] = (
    fusion["excel_severity_score"]
    .rank(
        ascending=False,
        method="dense",
    )
    .astype(int)
)

fusion["model_rank"] = (
    fusion["fusion_event_mean_expected_class_0_to_4"]
    .rank(
        ascending=False,
        method="dense",
    )
    .astype(int)
)

fusion["rank_difference"] = (
    fusion["model_rank"]
    - fusion["excel_rank"]
)

rank_accuracy = (
    (
        fusion["excel_rank"]
        ==
        fusion["model_rank"]
    )
    .mean()
    * 100
)

top_accuracy = float(
    fusion.loc[
        fusion["excel_rank"].idxmin(),
        "model_rank",
    ]
    == 1
)

bottom_accuracy = float(
    fusion.loc[
        fusion["excel_rank"].idxmax(),
        "model_rank",
    ]
    == fusion["model_rank"].max()
)

# -----------------------------------------------------------------------------
# Metrics table
# -----------------------------------------------------------------------------

fusion_metrics = pd.DataFrame(
    {
        "Metric": [
            "Pearson correlation",
            "Spearman correlation",
            "Kendall Tau",
            "MAE",
            "RMSE",
            "R²",
            "Rank accuracy (%)",
            "Top event accuracy",
            "Weakest event accuracy",
        ],
        "Value": [
            pearson_r,
            spearman_rho,
            kendall_tau,
            mae,
            rmse,
            r2,
            rank_accuracy,
            top_accuracy,
            bottom_accuracy,
        ],
    }
)

# -----------------------------------------------------------------------------
# Save
# -----------------------------------------------------------------------------

fusion_metrics.to_csv(
    OUTPUT_DIR / "fusion_validation_metrics.csv",
    index=False,
)

fusion.to_csv(
    OUTPUT_DIR / "fusion_event_rankings.csv",
    index=False,
)

# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

print()
print(fusion_metrics.to_string(index=False))

print()
print("=" * 80)
print("EVENT RANKING")
print("=" * 80)

print()

print(
    fusion[
        [
            "excel_event_period",
            "excel_severity_score",
            "fusion_event_mean_expected_class_0_to_4",
            "excel_rank",
            "model_rank",
            "rank_difference",
        ]
    ].to_string(index=False)
)

print()

print("Saved:")
print(OUTPUT_DIR / "fusion_validation_metrics.csv")
print(OUTPUT_DIR / "fusion_event_rankings.csv")

NEW MULTI-HORIZON MODEL VALIDATION

                Metric     Value
   Pearson correlation  0.217548
  Spearman correlation  0.500000
           Kendall Tau  0.333333
                   MAE  1.425106
                  RMSE  1.834502
                    R² -1.163471
     Rank accuracy (%) 33.333333
    Top event accuracy  0.000000
Weakest event accuracy  1.000000

EVENT RANKING

excel_event_period  excel_severity_score  fusion_event_mean_expected_class_0_to_4  excel_rank  model_rank  rank_difference
         2015–2016                     4                                 1.703980           2           1               -1
         2018–2019                     2                                 1.079254           3           3                0
         2023–2024                     5                                 1.099957           1           2                1

Saved:
c:\Projects\Infer RozviDrought\data\validation\outputs\fusion_validation_metrics.csv
c:\Projects\Infer RozviDrought\da

In [12]:
# =============================================================================
# Cell 8
# New Multi-Horizon Model Chronology Validation
#
# Purpose
# --------
# Assess whether the new model captures the temporal evolution of each drought
# event using the monthly summaries.
# =============================================================================

print("=" * 80)
print("NEW MODEL MONTHLY CHRONOLOGY")
print("=" * 80)

monthly = monthly_event_df.copy()

monthly["yyyymm"] = pd.to_numeric(monthly["yyyymm"])

results = []

for _, event in excel_df.iterrows():

    event_months = monthly.loc[
        (monthly["yyyymm"] >= event["event_start_yyyymm"])
        &
        (monthly["yyyymm"] <= event["event_end_yyyymm"])
    ].copy()

    if event_months.empty:
        continue

    peak_idx = event_months[
        "fusion_mean_expected_class_0_to_4"
    ].idxmax()

    peak_month = int(
        event_months.loc[
            peak_idx,
            "yyyymm",
        ]
    )

    peak_class = float(
        event_months.loc[
            peak_idx,
            "fusion_mean_expected_class_0_to_4",
        ]
    )

    mean_class = float(
        event_months[
            "fusion_mean_expected_class_0_to_4"
        ].mean()
    )

    persistence = float(
        (
            event_months[
                "fusion_mean_expected_class_0_to_4"
            ]
            >= mean_class
        ).mean()
    )

    first_month = int(
        event_months["yyyymm"].min()
    )

    last_month = int(
        event_months["yyyymm"].max()
    )

    peak_offset = len(
        event_months.loc[
            event_months["yyyymm"] < peak_month
        ]
    )

    results.append(
        {
            "event_key": event["event_key"],
            "event_period": event["event_period_raw"],
            "excel_severity": event["excel_severity_score"],
            "months_available": len(event_months),
            "first_month": first_month,
            "last_month": last_month,
            "peak_month": peak_month,
            "peak_offset_months": peak_offset,
            "mean_expected_class": mean_class,
            "peak_expected_class": peak_class,
            "persistence_score": persistence,
        }
    )

fusion_chronology = pd.DataFrame(results)

fusion_chronology.to_csv(
    OUTPUT_DIR / "fusion_monthly_chronology.csv",
    index=False,
)

print()
print(fusion_chronology.to_string(index=False))

print()

print("Saved:")
print(OUTPUT_DIR / "fusion_monthly_chronology.csv")

NEW MODEL MONTHLY CHRONOLOGY

               event_key event_period  excel_severity  months_available  first_month  last_month  peak_month  peak_offset_months  mean_expected_class  peak_expected_class  persistence_score
excel_event_15_2015–2016    2015–2016               4                 5       201510      201602      201512                   2             1.703980             2.328118           0.400000
excel_event_16_2018–2019    2018–2019               2                 8       201810      201905      201811                   1             1.079254             1.443205           0.500000
excel_event_17_2023–2024    2023–2024               5                 6       202310      202403      202311                   1             1.099957             2.008362           0.333333

Saved:
c:\Projects\Infer RozviDrought\data\validation\outputs\fusion_monthly_chronology.csv


In [13]:
# =============================================================================
# Cell 9
# Final Model Comparison Scorecard
#
# Purpose
# --------
# Compare RozviDrought and the new multi-horizon model using identical
# evaluation metrics and chronology statistics.
# =============================================================================

print("=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)

# -----------------------------------------------------------------------------
# Validation metrics
# -----------------------------------------------------------------------------

rv_metrics = pd.read_csv(
    OUTPUT_DIR / "rozvidrought_validation_metrics.csv"
).rename(
    columns={"Value": "RozviDrought"}
)

fusion_metrics = pd.read_csv(
    OUTPUT_DIR / "fusion_validation_metrics.csv"
).rename(
    columns={"Value": "Fusion_V2"}
)

scorecard = rv_metrics.merge(
    fusion_metrics,
    on="Metric",
)

scorecard["Difference"] = (
    scorecard["Fusion_V2"]
    - scorecard["RozviDrought"]
)

# -----------------------------------------------------------------------------
# Chronology comparison
# -----------------------------------------------------------------------------

chrono = (
    chronology_df.rename(
        columns={
            "mean_class": "rozvidrought_mean_class",
            "peak_class": "rozvidrought_peak_class",
            "peak_month": "rozvidrought_peak_month",
            "persistence_score": "rozvidrought_persistence",
        }
    )
    .merge(
        fusion_chronology.rename(
            columns={
                "mean_expected_class": "fusion_mean_class",
                "peak_expected_class": "fusion_peak_class",
                "peak_month": "fusion_peak_month",
                "persistence_score": "fusion_persistence",
            }
        ),
        on="event_key",
        suffixes=("_rv", "_fusion"),
    )
)

chrono = chrono[
    [
        "event_period_rv",
        "excel_severity_rv",
        "rozvidrought_mean_class",
        "fusion_mean_class",
        "rozvidrought_peak_class",
        "fusion_peak_class",
        "rozvidrought_peak_month",
        "fusion_peak_month",
        "rozvidrought_persistence",
        "fusion_persistence",
    ]
].rename(
    columns={
        "event_period_rv": "event_period",
        "excel_severity_rv": "excel_severity",
    }
)

# -----------------------------------------------------------------------------
# Save
# -----------------------------------------------------------------------------

scorecard_path = (
    OUTPUT_DIR
    / "final_model_scorecard.csv"
)

chrono_path = (
    OUTPUT_DIR
    / "final_model_chronology_comparison.csv"
)

scorecard.to_csv(
    scorecard_path,
    index=False,
)

chrono.to_csv(
    chrono_path,
    index=False,
)

# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

print()
print("=" * 80)
print("VALIDATION SCORECARD")
print("=" * 80)
print(scorecard.to_string(index=False))

print()
print("=" * 80)
print("CHRONOLOGY COMPARISON")
print("=" * 80)
print(chrono.to_string(index=False))

print()

winner = []

for _, row in scorecard.iterrows():

    metric = row["Metric"]

    rv = row["RozviDrought"]
    fv = row["Fusion_V2"]

    if metric in [
        "MAE",
        "RMSE",
    ]:
        better = "Fusion V2" if fv < rv else "RozviDrought"

    else:
        better = "Fusion V2" if fv > rv else "RozviDrought"

    winner.append(
        {
            "Metric": metric,
            "Better Model": better,
        }
    )

winner = pd.DataFrame(winner)

print("=" * 80)
print("BETTER MODEL PER METRIC")
print("=" * 80)
print(winner.to_string(index=False))

print()

print("Saved:")
print(scorecard_path)
print(chrono_path)

FINAL MODEL COMPARISON

VALIDATION SCORECARD
                Metric  RozviDrought  Fusion_V2  Difference
   Pearson correlation     -0.272909   0.217548    0.490457
  Spearman correlation     -0.500000   0.500000    1.000000
           Kendall Tau     -0.333333   0.333333    0.666667
                   MAE      2.523783   1.425106   -1.098678
                  RMSE      2.819558   1.834502   -0.985055
                    R²     -4.110654  -1.163471    2.947183
     Rank accuracy (%)      0.000000  33.333333   33.333333
    Top event accuracy      0.000000   0.000000    0.000000
Weakest event accuracy      0.000000   1.000000    1.000000

CHRONOLOGY COMPARISON
event_period  excel_severity  rozvidrought_mean_class  fusion_mean_class  rozvidrought_peak_class  fusion_peak_class  rozvidrought_peak_month  fusion_peak_month  rozvidrought_persistence  fusion_persistence
   2015–2016               4                 0.155757           1.703980                 0.757199           2.328118         

In [14]:
# =============================================================================
# Export RozviDrought pixel predictions to CSV
# =============================================================================

from pathlib import Path
import pandas as pd

DATA_DIR = Path(r"C:\Projects\Infer RozviDrought\data\validation")

parquet_file = DATA_DIR / "rozvidrought_spi3_pixel_predictions.parquet"
csv_file = DATA_DIR / "rozvidrought_spi3_pixel_predictions.csv"

df = pd.read_parquet(parquet_file)

df.to_csv(
    csv_file,
    index=False,
)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Saved:")
print(csv_file)

Rows: 508725
Columns: 12
Saved:
C:\Projects\Infer RozviDrought\data\validation\rozvidrought_spi3_pixel_predictions.csv
